# Eval Metrics Part → Second Leg of Hypothesis Tests (**Metrics Engine**)

In **Leg 2**, we stop “creating signals” and start **measuring performance**.

The key idea is simple:

> If Leg 1 tells us **where** a hypothesis applies (events) and **what it predicts**,  
> Leg 2 tells us **how well it actually works**.

- This notebook exists because we want **one consistent evaluation system** for *all* hypotheses, instead of writing different metric code every time. 
- Given an event definition and (optional) predictions, compute **all evaluation metrics** in a standardized way.

You can think of `eval_metrics.ipynb` as a reusable **metrics calculator**.


## The 3-Leg Pipeline (How the whole system is organized)

### **Leg 1 — prediction.ipynb (Event Factory)** 
**Goal:** Create new columns that mark **where** each hypothesis applies and **what the rule predicts** (if applicable).

- We do **NOT** prove anything here.
- We do **NOT** compute p-values or draw conclusions here.
- We only build:
  - **event masks** (which rows are “in the hypothesis world”)
  - **baseline rule predictions** (optional for some hypotheses)

**Output pattern (typical):**
- `is_Hx` → 0/1 mask for “this row is a valid event for Hx”
- `pred_Hx_*` → rule-based prediction (if the hypothesis is directional)
- additional tags (if the hypothesis is grouping / moderator)

Some hypotheses are already fully defined by existing features and labels → they may pass through this leg with minimal work.

### **Leg 2 — eval_metrics.ipynb (Metrics Engine)** --> WE ARE IN HERE !!!
**Goal:** Given an event definition and predictions, compute all **evaluation metrics** consistently.

Think of this as a reusable “calculator”:

- Input: **(mask, y_true, y_pred)**  
- Output: metrics such as:
  - **hit-rate**
  - **precision / recall / F1** (for rule predictions)
  - **p-value** (example: test `hit-rate > 0.5`)
  - **effect size** (example: signed returns, distance reduction)
  - reusable slice logic (IB width, gap alignment, etc.)

### **Leg 3 — hypothesis_tests.ipynb (Report + Decision Layer)**
**Goal:** Present results in a clean, viewer-friendly form and state decisions clearly.

This is where we:
- show tables/figures
- explain measurement choices (especially for “reversion” type ideas)
- decide:
  - **Reject null hypothesis** or
  - **Fail to reject null hypothesis**
based on the metrics and p-values from Leg 2.


## What we feed into the Metrics Engine → Inputs

For any hypothesis (H1–H5), we only need a small set of objects:

- **`mask`**  
  A boolean or 0/1 filter that selects the rows we are evaluating  

- **`y_true`**  
  The true future outcome we want to evaluate against  

- **`y_pred`** *(only if the hypothesis produces a rule prediction)*  
  The rule-based predicted outcome  


## What the Metrics Engine returns → Outputs

Depending on the hypothesis type, `eval_metrics.ipynb` can compute metrics like:

- **Hit-rate (accuracy)**  
  “On the masked event rows, what fraction of predictions were correct?”

- **Precision / Recall / F1** *(for direction/rule hypotheses)*  
  Extra classification quality metrics, especially useful when class balance matters.

- **p-value (statistical significance)**  
  Examples:
  - Test whether **hit-rate > 0.5** (better than random guessing)
  - Test whether **mean return ≠ 0** (or > 0 / < 0 depending on hypothesis)

- **Effect size (economic/statistical magnitude)**  
  Examples:
  - mean/median **signed returns** (`ret15`, `ret30`)
  - “distance reduction” style outcomes for reversion hypotheses  
    (example: did `|close_f15 - ib_mid|` shrink vs now?)

- **Reusable “slice” logic (conditional analysis)**  
  The same hypothesis can be evaluated under conditions such as:
  - **IB width regime:** narrow vs wide (`is_H6_narrow`, `is_H6_wide`)
  - **Gap alignment:** aligned vs not (`is_H7_align`)
  - other sensitivity filters (like whipsaw vs no-whipsaw)

    * This allows clean comparisons like:
      - “Does H2 work better on wide-IB days?”
      - “Does H4 improve when whipsaw is excluded?”
      - “Are returns larger when gap alignment is present?”

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

### **H1: Reversion in between state**

#### **(a) Mask (Which rows do we evaluate?)**

H1 should **not** be evaluated on every 1-minute candle.  
We only want rows that are:

1. **Actually valid H1 events**, and
2. **Inside our analysis universe**, and
3. **Safe for label creation** (so the “future” values exist and are not broken)

That is why we combine multiple masks:

- **`is_H1`**  
  Created in **Leg 1 (Prediction)**. This is the main H1 event definition.  
  It answers:  
  **“Is this 1-minute candle in the H1 setup world?”**

- **`is_analysis`**  
  A global filter that defines the rows we allow in our statistical evaluation.  
  It answers:  
  **“Is this candle in our official analysis/testing area?”**  
  (Example use: excluding premarket, illiquid sections, or anything outside our study design.)

- **`is_labelwin`**  
  Created in **`labels.ipynb`** to ensure the row is safe for future-label logic.  
  It answers:  
  **“Do we have a reliable future window after this candle to compute 15/30-minute outcomes?”**  
  This prevents broken labels near the end of the day (or missing future bars).

**Practical final mask for H1 evaluation:**
- Evaluate only rows where all conditions hold:
  - `is_H1 == 1`
  - `is_analysis == 1`
  - `is_labelwin == 1`

This guarantees we test H1 only where it is **defined**, **allowed**, and **measurable**.


#### **(b) `y_true` (What actually happened?)**

`y_true` is the **real-world outcome** from the market (our ground truth).

For the directional version of H1, we use:

- **`dir15`** → the true direction **15 minutes later**
- **`dir30`** → the true direction **30 minutes later**

These columns answer:

- “After this candle, did SPY end up **higher (1)** or **lower (0)** after 15/30 minutes?”

So `y_true` is the reality we want to match.


#### **(c) `y_pred` (What does H1 predict should happen?)**

`y_pred` is the output of our hypothesis rule — what H1 *claims* should happen.

For H1, the rule-based baseline predictions (created in Leg 1) are:

- **`pred_H1_dir15`** → H1’s predicted direction for the next 15 minutes
- **`pred_H1_dir30`** → H1’s predicted direction for the next 30 minutes

These represent:

- “In the H1 setup world, price should move **toward the middle**, which we approximate as an up/down prediction.”


In [2]:
PROJECT_ROOT = Path("..").resolve()

DATA_CACHE = PROJECT_ROOT / "data" / "cache"

CACHE_FILE = DATA_CACHE / "spy_1min_et_with_H1_events.csv"

df_eval1 = pd.read_csv(CACHE_FILE, parse_dates=['datetime'])

df_eval1.head()

,datetime,high,low,close,Volume,hl_pct,hl5,hl15,trend_score_m30,ib_high,...,cross_av_od_last5,close_f15,close_f30,ret15,ret30,dir15,dir30,is_H1,pred_H1_dir15,pred_H1_dir30
0,2025-09-08 09:30:00,648.86,648.24,648.260,141588,0.000956,NaN,NaN,NaN,649.06,...,0,648.42,648.24,0.000247,-0.000031,1,0,0,NaN,NaN
1,2025-09-08 09:31:00,648.45,648.15,648.270,42118,0.000463,NaN,NaN,NaN,649.06,...,0,648.28,647.97,0.000015,-0.000463,1,0,0,NaN,NaN
2,2025-09-08 09:32:00,648.46,648.10,648.260,37143,0.000555,NaN,NaN,NaN,649.06,...,0,648.11,648.27,-0.000231,0.000015,0,1,0,NaN,NaN
3,2025-09-08 09:33:00,648.47,648.23,648.400,42231,0.000370,NaN,NaN,NaN,649.06,...,0,648.57,648.24,0.000262,-0.000247,1,0,0,NaN,NaN
4,2025-09-08 09:34:00,648.68,648.32,648.665,23659,0.000555,0.00058,NaN,NaN,649.06,...,0,648.66,648.29,-0.000008,-0.000578,0,0,0,NaN,NaN


In [3]:
# 1) H1 evaluation mask
mask_H1_eval = (
    (df_eval1["is_H1"] == 1) &      
    (df_eval1["is_analysis"] == 1) & 
    (df_eval1["is_labelwin"] == 1)   
)

# 2) Real results (y_true) -> what happened in the market?
y_true_H1_dir15 = df_eval1.loc[mask_H1_eval, "dir15"]
y_true_H1_dir30 = df_eval1.loc[mask_H1_eval, "dir30"]

# 4) H1's prediction (y_pred) -> What should happen in H1's world?
y_pred_H1_dir15 = df_eval1.loc[mask_H1_eval, "pred_H1_dir15"]
y_pred_H1_dir30 = df_eval1.loc[mask_H1_eval, "pred_H1_dir30"]

print("Total rows :", len(df_eval1))
print("H1 eval mask's selections :", mask_H1_eval.sum())

print("First 5 observations for 15 min (y_true vs y_pred):")
print(pd.DataFrame({
    "dir15": y_true_H1_dir15.head(),
    "pred_H1_dir15": y_pred_H1_dir15.head()
}))

print("First 5 observations for 30 min (y_true vs y_pred):")
print(pd.DataFrame({
    "dir30": y_true_H1_dir30.head(),
    "pred_H1_dir30": y_pred_H1_dir30.head()
}))


Total rows : 21450
H1 eval mask's selections : 326
First 5 observations for 15 min (y_true vs y_pred):
     dir15  pred_H1_dir15
134      1            0.0
135      1            0.0
144      0            0.0
153      0            0.0
154      0            0.0
First 5 observations for 30 min (y_true vs y_pred):
     dir30  pred_H1_dir30
134      0            0.0
135      0            0.0
144      0            0.0
153      0            0.0
154      0            0.0


In [4]:
import math

# --------------------------
# 0) Setup
# --------------------------
df = df_eval1 # our code includes lots of pandas operations so we need to represent it shortly

df["datetime"] = pd.to_datetime(df["datetime"], errors="coerce")
df["date"] = df["datetime"].dt.date

# --------------------------
# 1) Binomial p-value
# --------------------------

def _log_choose(n, k): # in our prediction of sample size n events, k events must be true
    return math.lgamma(n+1) - math.lgamma(k+1) - math.lgamma(n-k+1)

def _log_binom_pmf(k, n, p): # log version of the probability of seeing k events as true with exact X = k
    if p <= 0 or p >= 1: # we don't want to consider log(0) scenarios
        return -math.inf
    return _log_choose(n, k) + k*math.log(p) + (n-k)*math.log(1-p)

def _logsumexp(log_terms): # our probabilities may be too low but I don't want to lose them
    # so I got largest of them and scale everything according to that number
    m = max(log_terms)
    if m == -math.inf:
        return -math.inf
    return m + math.log(sum(math.exp(t-m) for t in log_terms))

def binom_cdf(k, n, p): # The most k true events seen probability with 
    # P(X <= k), also can be considered as "acceptance region"
    if k < 0: return 0.0
    if k >= n: return 1.0
    logs = [_log_binom_pmf(i, n, p) for i in range(0, k+1)]
    return float(math.exp(_logsumexp(logs)))

def binom_sf(k_minus_1, n, p): # right tail p-value (probability) for clearly determining the rejection area
    # very important tool in hypothesis decisions
    # P(X >= k) = 1 - P(X <= k-1)
    return float(1.0 - binom_cdf(k_minus_1, n, p))

# --------------------------
# 2) Main metrics function (hit/precision/recall/F1 + signed return + p-value)
# --------------------------
def metrics_from_series(y_true_s, y_pred_s, ret_s, p0=0.5): # we assigned p0 as 0.5 because we need to understand 
    # whether our hypothesis shows us something better than a coin-flip

    m = (~y_true_s.isna()) & (~y_pred_s.isna()) # from same index clear all NaNs
    yt = y_true_s.loc[m].astype(int).to_numpy()
    yp = y_pred_s.loc[m].astype(int).to_numpy()
    rr = ret_s.loc[m].astype(float).to_numpy()

    # if there is no column to evaluate, we don't need to waste our time
    N = int(len(yt))
    if N == 0:
        return {"N": 0}

    # Hit-rate calculation and evaluate the true predictions
    correct = (yt == yp)
    k = int(correct.sum())
    hit_rate = k / N

    # Confusion matrix with stating prediction vs. reality
    tp = int(((yp==1) & (yt==1)).sum()) # prediction -> up, reality -> up
    fp = int(((yp==1) & (yt==0)).sum()) # prediction -> up, reality -> down
    fn = int(((yp==0) & (yt==1)).sum()) # prediction -> down, reality -> up
    tn = int(((yp==0) & (yt==0)).sum()) # prediction -> down, reality -> down

    precision = tp/(tp+fp) if (tp+fp)>0 else np.nan # how many my predictions of up is really up in reality
    recall    = tp/(tp+fn) if (tp+fn)>0 else np.nan # how many real ups I got from my predictions
    f1 = (2*precision*recall/(precision+recall)) if (not np.isnan(precision) and not np.isnan(recall) and (precision+recall)>0) else np.nan
    # harmonic average of both variables

    # p-values under H0: X ~ Binomial(N, p0), X = #correct
    p_greater = binom_sf(k-1, N, p0)         # P(X >= k), better than 50%?
    p_less    = binom_cdf(k,   N, p0)        # P(X <= k), worse than 50%?
    p_two     = float(min(1.0, 2*min(p_greater, p_less))) # two tailed hypothesis graph with not exceeding 1.

    signed_r = np.where(yp==1, rr, -rr) # consider if our prediction shows up take return as +, if not take return as -
    mean_sr = float(np.nanmean(signed_r)) # if I am opening positions at just prediction's side what would be my mean return
    med_sr  = float(np.nanmedian(signed_r)) # if I am opening positions at just prediction's side what would be my median return

    p_true = float(yt.mean()) # the ratio of "up" values; up means 1, down means 0 we consider both of them
    # if one of them is majority we will measure its direct ratio of that
    majority_acc = float(max(p_true, 1-p_true)) # also we are considering which one is the most successful p_value
    # if up's (1) are majority we take up's, otherwise down's (0)

    return {
        "N": N, "k_correct": k, "hit_rate": hit_rate,
        "tp": tp, "fp": fp, "fn": fn, "tn": tn,
        "precision": precision, "recall": recall, "f1": f1,
        "pval_greater(>p0)": p_greater,
        "pval_less(<p0)": p_less,
        "pval_two_sided": p_two,
        "mean_signed_return": mean_sr,
        "median_signed_return": med_sr,
        "y_true_up_rate": p_true,
        "majority_baseline_acc": majority_acc,
    } # reporting in a dictionary format


# If value returns NaN, directly says "not calculated"
def _fmt(x):
    return "not calculated" if (x is None or (isinstance(x,float) and np.isnan(x))) else x
def pretty(d):
    return {k:_fmt(v) for k,v in d.items()} if isinstance(d, dict) else d

# --------------------------
# 3) Bootstrap (daily basis)
# --------------------------
# In general, the logic is very simple:
# - Selected days in our sample by mask considered as blocks 
# - We calculate bootstrap confidence interval from that.

def bootstrap_by_day(mask, y_true_col, y_pred_col, ret_col, B=2000, seed=7):
    # Just taking our needed day columns, with dropping every NaN columns because we won't need them
    sub = df.loc[mask, ["date", y_true_col, y_pred_col, ret_col]].dropna(subset=[y_true_col, y_pred_col]).copy()
    if sub.empty:
        return None

    # row based numpy operations
    # we convert every needed things as arrays to make our bootstrap operations faster
    day = sub["date"].to_numpy()
    yt  = sub[y_true_col].astype(int).to_numpy()
    yp  = sub[y_pred_col].astype(int).to_numpy()
    rr  = sub[ret_col].astype(float).to_numpy()
    signed_r = np.where(yp==1, rr, -rr)

    # gün bazında topla (bootstrap'ta concat yok)
    days, inv = np.unique(day, return_inverse=True)
    n_days = len(days)

    # outputting daily basis statistics
    # inv -> tells directly which index corresponds to which trading day 
    # np.bincount(inv) -> how many rows available for each day
    N_d  = np.bincount(inv)
    k_d  = np.bincount(inv, weights=(yt==yp).astype(int))
    tp_d = np.bincount(inv, weights=((yp==1)&(yt==1)).astype(int))
    fp_d = np.bincount(inv, weights=((yp==1)&(yt==0)).astype(int))
    fn_d = np.bincount(inv, weights=((yp==0)&(yt==1)).astype(int))
    sr_d = np.bincount(inv, weights=signed_r)
    # Our main logic is calculating everything in intraday and after summing all of them
    # This makes our operation much easier

    # in every bootstrap instance, our code needs to select n_days days from selecting in days for randomness
    rng = np.random.default_rng(seed)
    hit_list, f1_list, msr_list = [], [], []
    f1_nan = 0

    for _ in range(B):
        pick = rng.integers(0, n_days, size=n_days)  # replacement
        N  = int(N_d[pick].sum()) # if there is no row in selected day, our code needs to say directly this is NaN value
        if N == 0:
            hit_list.append(np.nan); f1_list.append(np.nan); msr_list.append(np.nan); f1_nan += 1
            continue

        # the summation of selected days summaries -> the summary of bootstrap world's summation
        k  = float(k_d[pick].sum())
        tp = float(tp_d[pick].sum())
        fp = float(fp_d[pick].sum())
        fn = float(fn_d[pick].sum())
        sr = float(sr_d[pick].sum())

        # saving that bootstrap summation hit-rate
        hit_list.append(k / N)

        # saving that bootstrap summation's F1
        prec = tp/(tp+fp) if (tp+fp)>0 else np.nan
        rec  = tp/(tp+fn) if (tp+fn)>0 else np.nan
        f1 = (2*prec*rec/(prec+rec)) if (not np.isnan(prec) and not np.isnan(rec) and (prec+rec)>0) else np.nan
        if np.isnan(f1): f1_nan += 1
        f1_list.append(f1)

        # when days are resampling, our code needs to save average signed return also
        msr_list.append(sr / N)

    # Now, we are dropping all NaNs and returning real Confidence Interval
    # with 2.5%, 50% and 97.5% quartiles
    def ci(arr):
        arr = np.array(arr, dtype=float)
        arr = arr[~np.isnan(arr)]
        if len(arr)==0:
            return (np.nan, np.nan, np.nan)
        return (float(np.quantile(arr,0.025)), float(np.quantile(arr,0.50)), float(np.quantile(arr,0.975)))

    # returning our bootstrap results dictionary
    return {
        "n_days": int(n_days), "B": int(B),
        "hit_ci(2.5,50,97.5)": ci(hit_list),
        "f1_ci(2.5,50,97.5)": ci(f1_list),
        "mean_signed_return_ci(2.5,50,97.5)": ci(msr_list),
        "f1_nan_rate": f1_nan / B
    }

# --------------------------
# 4) RUN (overall + slices + bootstrap) — summary outputs just for visualization and showing the code really works
# --------------------------
out_hypothesis1 = {}

# Overall (direkt senin oluşturduğun Series'lerle)
out_hypothesis1["overall_15"] = metrics_from_series(y_true_H1_dir15, y_pred_H1_dir15, df.loc[mask_H1_eval, "ret15"], p0=0.5)
out_hypothesis1["overall_30"] = metrics_from_series(y_true_H1_dir30, y_pred_H1_dir30, df.loc[mask_H1_eval, "ret30"], p0=0.5)

# Slice: ib_width_type (narrow/wide) — yine aynı tanım, sadece mask daraltılıyor
for val in ["narrow", "wide"]:
    m = mask_H1_eval & (df["ib_width_type"] == val)
    out_hypothesis1[f"{val}_15"] = metrics_from_series(df.loc[m,"dir15"], df.loc[m,"pred_H1_dir15"], df.loc[m,"ret15"], p0=0.5)
    out_hypothesis1[f"{val}_30"] = metrics_from_series(df.loc[m,"dir30"], df.loc[m,"pred_H1_dir30"], df.loc[m,"ret30"], p0=0.5)

# Bootstrap (day-by-day)
out_hypothesis1["bootstrap_overall_15"] = bootstrap_by_day(mask_H1_eval, "dir15", "pred_H1_dir15", "ret15", B=2000, seed=7)
out_hypothesis1["bootstrap_overall_30"] = bootstrap_by_day(mask_H1_eval, "dir30", "pred_H1_dir30", "ret30", B=2000, seed=7)

# Print
for k,v in out_hypothesis1.items():
    print("\n====================", k, "====================")
    print(pretty(v))



==================== overall_15 ====================
{'N': 326, 'k_correct': 135, 'hit_rate': 0.41411042944785276, 'tp': 68, 'fp': 111, 'fn': 80, 'tn': 67, 'precision': 0.37988826815642457, 'recall': 0.4594594594594595, 'f1': 0.41590214067278286, 'pval_greater(>p0)': 0.9992225940463264, 'pval_less(<p0)': 0.0011340343275836327, 'pval_two_sided': 0.0022680686551672653, 'mean_signed_return': -0.0002096181345698247, 'median_signed_return': -0.00010388184773285, 'y_true_up_rate': 0.4539877300613497, 'majority_baseline_acc': 0.5460122699386503}

==================== overall_30 ====================
{'N': 326, 'k_correct': 156, 'hit_rate': 0.4785276073619632, 'tp': 89, 'fp': 90, 'fn': 80, 'tn': 67, 'precision': 0.4972067039106145, 'recall': 0.5266272189349113, 'f1': 0.5114942528735632, 'pval_greater(>p0)': 0.7969302581294637, 'pval_less(<p0)': 0.2357889885645186, 'pval_two_sided': 0.4715779771290372, 'mean_signed_return': -0.00011292030256727929, 'median_signed_return': -6.624324343346144e-05